# 03 — H&E/IHC whole-slide alignment

This notebook uses the current registration API and demonstrates:

1. the pair-folder convention;
2. a scanner-free synthetic discovery dry run;
3. a real-data dry run;
4. ORB or VALIS registration with optional center-patch QC; and
5. result inspection.

Use `.[orb]` for contour/ORB alignment and streamed OME-TIFF export.
Use `.[valis]` for rigid/non-rigid VALIS registration.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


## Required directory and filename contract

For a CD8-specific run with `reference_name="he"` and
`moving_name="cd8"`:

```text
data/pairs/
└── CD8/
    ├── he/
    │   ├── Sample_0001_he.svs
    │   └── Sample_0002_he.svs
    └── cd8/
        ├── Sample_0001_cd8.svs
        └── Sample_0002_cd8.svs
```

Filenames must share the same sample ID before the terminal role token.
Sample IDs may contain underscores. The role in the filename must match
its containing subfolder, case-insensitively.

For several biomarkers, either run once per marker with marker-specific
moving names, or use a generic `moving_name="ihc"` in every pair folder.


In [ ]:
import os

from PIL import Image

from rocqipath.registration import AlignmentConfig, run_alignment

demo_root = Path(
    os.environ.get(
        "ROCQIPATH_NOTEBOOK_DEMO_DIR",
        str(PROJECT_ROOT / "notebook_demo_outputs"),
    )
)
synthetic_input = demo_root / "alignment_discovery" / "pairs"
synthetic_output = demo_root / "alignment_discovery" / "results"

for role, color in (("he", (205, 145, 180)), ("cd8", (180, 120, 70))):
    path = synthetic_input / "CD8" / role / f"Case_With_Underscores_{role}.tif"
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.new("RGB", (32, 32), color).save(path, format="TIFF")

demo_cfg = AlignmentConfig(
    input_dir=str(synthetic_input),
    output_dir=str(synthetic_output),
    pair_folders=["CD8"],
    reference_name="he",
    moving_name="cd8",
    alignment_method="orb",
    dry_run=True,
)
dry_results = run_alignment(demo_cfg)
print(f"Dry-run result objects: {len(dry_results)}")
print(
    "Current behavior: dry-run logs discovered pairs but intentionally "
    "returns an empty list."
)


In [ ]:
PAIRS_ROOT = DATA_ROOT / "pairs"
OUTPUT_ROOT = RESULTS_ROOT
MARKER = "CD8"

REFERENCE_NAME = "he"
MOVING_NAME = "cd8"
ALIGNMENT_METHOD = "orb"  # "orb" or "valis"
TARGET_MAGNIFICATION = 20.0
ALIGNED_WSI_LEVEL = 0

REFERENCE_SOURCE_MAGNIFICATION = None
MOVING_SOURCE_MAGNIFICATION = None

RUN_REAL_DRY_RUN = False
RUN_ALIGNMENT = False


In [ ]:
real_cfg = AlignmentConfig(
    input_dir=str(PAIRS_ROOT),
    output_dir=str(OUTPUT_ROOT),
    pair_folders=[MARKER],
    reference_name=REFERENCE_NAME,
    moving_name=MOVING_NAME,
    alignment_method=ALIGNMENT_METHOD,
    target_magnification=TARGET_MAGNIFICATION,
    reference_source_magnification=REFERENCE_SOURCE_MAGNIFICATION,
    moving_source_magnification=MOVING_SOURCE_MAGNIFICATION,
    max_physical_field_ratio=2.0,
    patch_size=1024,
    grid_density=8,
    aligned_wsi_level=ALIGNED_WSI_LEVEL,
    qc_enabled=True,
    qc_patch_size=1024,
    qc_dpi=300,
    dry_run=True,
)

print("Resolved filename regex:")
print(real_cfg.filename_pattern)
print("\nConfiguration:")
for label, value in real_cfg.describe():
    print(f"  {label:34s} {value}")


## Real-data discovery dry run

Run this before installing a registration backend or committing hours to a
batch. It validates directory discovery and filename pairing. The current
implementation logs successfully discovered dry-run pairs but returns no
`AlignedCaseResult` objects, so assess the log rather than list length.


In [ ]:
if RUN_REAL_DRY_RUN:
    if not PAIRS_ROOT.is_dir():
        raise FileNotFoundError(PAIRS_ROOT)
    _ = run_alignment(real_cfg)
else:
    print("Set RUN_REAL_DRY_RUN=True after verifying the Parameters cell.")


## Run ORB or VALIS

- **ORB** is a good first choice for similar serial sections and has a
  smaller dependency footprint.
- **VALIS** adds rigid/non-rigid registration and may be more robust for
  larger tissue deformation, but it is heavier and slower.

QC does not determine scientific validity by itself. Review tissue edges,
vessels, glands, and other morphology across the slide—not only the center.


In [ ]:
from dataclasses import replace

if RUN_ALIGNMENT:
    if not PAIRS_ROOT.is_dir():
        raise FileNotFoundError(PAIRS_ROOT)

    aligned_results = run_alignment(replace(real_cfg, dry_run=False))

    for result in aligned_results:
        print(f"Case        : {result.case.case_id}")
        print(f"Aligned WSI : {result.aligned_moving_path}")
        print(f"Tissue grids: {len(result.valid_grids)}")
        print()
else:
    aligned_results = []
    print("Set RUN_ALIGNMENT=True only after the dry run is correct.")


## Output contract

The output is:

```text
results/alignment/<case>/
├── <case>_aligned_moving.ome.tiff
├── <case>_aligned_moving_manifest.json
├── ... registration diagnostics ...
└── <case>_center_qc.png   # when QC is enabled
```

The case name combines the sample ID and lowercased pair-folder name, for
example `Sample_0001_cd8`.

`target_magnification` controls target-grid operations, but
`aligned_wsi_level` controls exported WSI resolution. With level 0,
the aligned file is at the reference slide's level-0 objective. The sidecar
manifest records that output magnification for downstream `SlideReader` use.


In [ ]:
alignment_root = OUTPUT_ROOT / "alignment"
aligned_files = (
    sorted(alignment_root.rglob("*.ome.tif*"))
    if alignment_root.is_dir()
    else []
)
qc_files = (
    sorted(alignment_root.rglob("*_center_qc.png"))
    if alignment_root.is_dir()
    else []
)

print(f"Aligned WSIs: {len(aligned_files)}")
for path in aligned_files[:10]:
    print(" ", path.relative_to(OUTPUT_ROOT))

print(f"\nCenter QC files: {len(qc_files)}")
for path in qc_files[:10]:
    print(" ", path.relative_to(OUTPUT_ROOT))


## Troubleshooting

- **No pair folders found**: check the pair folder and role subfolder names.
- **Filename did not match**: use terminal `_he` and `_cd8` role tokens.
- **Role mismatch**: a `_cd8` file must be inside the `cd8/` subfolder.
- **Physical field ratio error**: verify objective metadata/fallbacks and
  confirm the slides represent comparable tissue fields.
- **Poor ORB match**: try VALIS and inspect whether the tissue sections are
  genuinely homologous.
- **libvips/OpenSlide error**: install both native runtimes and restart the
  kernel.
